# SPS01–BBRES | Experimental hydraulic characterisation and collection windows

**Research question:** When does the PBS/ethanol system produce liposomes under sufficiently stable hydraulic conditions?

**Workflow:** 1. Inspect actual CSV pressure and syringe traces → 2. Calibrate hydraulic resistance from measured pressure and gravimetric flow → 3. Compare pulsations, transients and trapped-air effects → 4. Quantify dead volume and transport → 5. Identify candidate collection windows.

Every section follows **question → measurement → equations → plots → interpretation**. Synthetic/model outputs are labelled and never substituted for measured data. Pressure stability is a proxy for flow stability; it does not alone establish constant FRR.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import periodogram
from IPython.display import display
LOG_DIR=Path('.')  # Folder containing uProcess CSVs
LOG_PATTERN='uPscript*.csv*'
SYRINGE_CAPACITY_UL=80
Q_PBS_UL_MIN=2000.
Q_ETHANOL_UL_MIN=1000.
# Check sensor assignments experimentally before assigning labels
SENSOR_LABELS={'P12159':'P12159 (assignment unconfirmed)','P22200':'P22200 (assignment unconfirmed)'}

## 1. What happens during pumping?

**Measure:** uProcess pressure and syringe-position logs at several flow rates and intentional air volumes. **Plot the raw traces first** to identify refilling, startup, pressure plateaus and fluctuations. Syringe-position derivatives are diagnostics, **not direct chip flow measurements**.

**Plots:** pressure versus time, syringe position versus time, and position-derived flow. Sampling intervals are reported; fast pulsations above the CSV Nyquist frequency cannot be resolved.

In [ ]:
from pathlib import Path
import re
# LOG_DIR set above
# LOG_PATTERN set above

def load_uprocess(path):
    path=Path(path)
    first=path.open(encoding='utf-8-sig').readline().strip()
    df=pd.read_csv(path,sep=';' if first=='sep=;' else ',',skiprows=1 if first=='sep=;' else 0)
    df.columns=df.columns.str.strip()
    df=df.loc[:,~df.columns.str.startswith('Unnamed:')]
    df=df.rename(columns={'Time (s)':'time_s'})
    if 'time_s' not in df: raise ValueError('Time (s) column missing')
    for col in df:
        if col!='Time of day':df[col]=pd.to_numeric(df[col],errors='coerce')
    df=df.dropna(subset=['time_s']).sort_values('time_s').reset_index(drop=True)
    pressures=[c for c in df if '(kPa)' in c and 'Temp' not in c]
    volumes=[c for c in df if 'Volume (ul)' in c]
    return df,pressures,volumes

def show_log(path):
    df,pressures,volumes=load_uprocess(path)
    t=df.time_s.to_numpy();dt=np.diff(t);dt=dt[dt>0]
    stats={'file':Path(path).name,'rows':len(df),'duration_s':t[-1]-t[0],
           'dt_median_s':np.median(dt),'dt_min_s':dt.min(),'dt_max_s':dt.max(),
           'log_rate_Hz':1/np.median(dt)}
    fig,axs=plt.subplots(3,1,figsize=(10,9),sharex=True)
    for c in pressures:
        axs[0].plot(t,df[c],label=c)
        stats[c+' mean_kPa']=df[c].mean();stats[c+' max_kPa']=df[c].max()
    for c in volumes:
        axs[1].plot(t,df[c],label=c)
        axs[2].plot(t,-np.gradient(df[c].to_numpy(),t)*60,label=c)
    for ax,ylabel in zip(axs,['Pressure (kPa)','Syringe position (µL)','Position-derived flow (µL/min)']):
        ax.set_ylabel(ylabel);ax.legend(fontsize=8);ax.grid(alpha=.25)
    axs[-1].set_xlabel('Time (s)');fig.suptitle(Path(path).name)
    plt.tight_layout();plt.show()
    return df,pressures,volumes,stats

logs={};summaries=[]
for path in sorted(LOG_DIR.glob(LOG_PATTERN)):
    try:
        df,ps,vs,stats=show_log(path)
        logs[path.name]=(df,ps,vs);summaries.append(stats)
    except Exception as e: print('Skipped',path.name,e)
if summaries:display(pd.DataFrame(summaries).round(3))
else:print('No logs found. Put CSVs beside notebook or change LOG_DIR.')

### Compare different experimental conditions

Add the actual air volume and condition for each file; do **not** infer experimental conditions from filenames. Overlay pressure traces only as descriptive comparisons until conditions are matched.

In [ ]:
RUN_METADATA={
 # 'uPscript100022.csv.csv': {'air_uL':0,'label':'No intentional air'},
 # 'uPscript100023.csv.csv': {'air_uL':20,'label':'20 µL air'},
}
if logs:
    fig,ax=plt.subplots(figsize=(11,4))
    for name,(df,ps,vs) in logs.items():
        for col in ps:
            sid=re.search(r'P\d+',col)
            ax.plot(df.time_s,df[col],label=f'{RUN_METADATA.get(name,{}).get("label",name)} / {sid.group(0) if sid else col}')
    ax.set(xlabel='Time (s)',ylabel='Gauge pressure (kPa)',title='Pressure traces — compare observed transients')
    ax.legend(fontsize=7);ax.grid(alpha=.25);plt.show()

## 2. What is the hydraulic resistance?

**Measure:** For each independent steady-flow experiment, select a window on its own pressure trace and measure collected mass over a known duration. Calculate $Q_{grav}=m/(\rho\Delta t)$, then fit $\overline{\Delta P}=R_h Q_{grav}+P_0$. Use pressure difference across the hydraulic load, not simply gauge pressure if outlet pressure differs from ambient.

**Plot:** selected steady windows and mean pressure drop versus measured mean flow. This is the **only resistance-calibration section**. No resistance estimate is possible until gravimetric measurements are supplied.

In [ ]:
# REAL resistance calibration — OPTIONAL, based on Section 3 LabSmith logs.
# One row per independent steady-state experiment; do NOT infer Q from the pump command.
# Set measured mass, density and collection times for each run.
# Pressure column must match a sensor name discovered by load_uprocess().
GRAVIMETRIC_RUNS = [
    # dict(filename='uPscript100023.csv.csv', sensor='P12159',
    #      steady_start_s=10, steady_end_s=20, collected_mass_g=0.05,
    #      collection_s=10, density_g_mL=0.997),
]

def fit_resistance_from_logs(specs):
    rows=[]
    for spec in specs:
        df,ps,_=load_uprocess(LOG_DIR / spec['filename'])
        sensor_matches=[col for col in ps if spec['sensor'] in col]
        if len(sensor_matches)!=1:
            raise ValueError(f"Sensor {spec['sensor']} not uniquely found in {spec['filename']}: {ps}")
        mask=df.time_s.between(spec['steady_start_s'],spec['steady_end_s'])
        if mask.sum()<3:raise ValueError('Insufficient samples in selected pressure window')
        p_mean=float(df.loc[mask,sensor_matches[0]].mean())
        q=(spec['collected_mass_g']/spec['density_g_mL'])*1000/spec['collection_s']
        if q<=0:raise ValueError('Measured flow must be positive')
        rows.append(dict(file=spec['filename'],sensor=spec['sensor'],P_mean_kPa=p_mean,Q_measured_uL_s=q))
    result=pd.DataFrame(rows)
    if len(result)<3:
        print('At least 3 independent flow conditions needed for a linear R fit.')
        return result
    slope,intercept=np.polyfit(result.Q_measured_uL_s,result.P_mean_kPa,1)
    fig,ax=plt.subplots(figsize=(7,4))
    ax.scatter(result.Q_measured_uL_s,result.P_mean_kPa,label='Measured runs')
    x=np.linspace(0,result.Q_measured_uL_s.max()*1.1,100)
    ax.plot(x,slope*x+intercept,label=f'R = {slope:.3f} kPa·s/µL')
    ax.set(xlabel='Measured mean flow (µL/s)',ylabel='Mean pressure (kPa)',title='Hydraulic resistance from measured runs')
    ax.legend();ax.grid(alpha=.25);plt.show()
    print(f'R = {slope:.4f} kPa·s/µL; intercept = {intercept:.3f} kPa')
    return result

if GRAVIMETRIC_RUNS:
    display(fit_resistance_from_logs(GRAVIMETRIC_RUNS))
else:
    print('No gravimetric calibration runs entered; pressure logs still analysed above.')


## 3. How do air volume, pulsations and transients interact?

**Measure:** Repeat matched runs at 0, 20, 50 and 100 µL initial trapped air. Select the same stage of discharge for each. Compare steady-window pressure SD/range **separately** from post-refill settling time. Do not call low-frequency CSV fluctuations fast pump pulsations.

**Model:** $C_g(P)\approx V_g(P)/P_{abs}$; $\tau=R_h(C_{system}+C_g)$; $|H(f)|=[1+(2\pi f\tau)^2]^{-1/2}$. Increasing gas compliance may reduce oscillations but lengthen transients and consume stroke volume. The next plots are **illustrative** until $R_h$, compliance and operating pressure are measured.

In [ ]:
# Matched experimental comparison (requires documented metadata and windows)
STEADY_WINDOWS={
 # 'uPscript100022.csv.csv':(12,22),
}
rows=[]
for name,(df,ps,vs) in logs.items():
    if name not in RUN_METADATA or name not in STEADY_WINDOWS:continue
    air=RUN_METADATA[name].get('air_uL')
    if air is None:continue
    lo,hi=STEADY_WINDOWS[name]
    for col in ps:
        a=df.loc[df.time_s.between(lo,hi),col].dropna()
        if len(a)<4:continue
        sid=re.search(r'P\d+',col).group(0)
        rows.append(dict(file=name,sensor=sid,air_uL=air,mean_kPa=a.mean(),sd_kPa=a.std(ddof=0),range_kPa=a.max()-a.min()))
comparison=pd.DataFrame(rows)
if comparison.empty: print('Add measured air volumes and steady windows to compare attenuation.')
else:
    display(comparison.round(3))
    fig,ax=plt.subplots(figsize=(7,4))
    for sensor,g in comparison.groupby('sensor'):
        ax.plot(g.air_uL,g.sd_kPa,'o-',label=sensor)
    ax.set(xlabel='Initial air (µL)',ylabel='Pressure SD in steady window (kPa)',title='Experimental fluctuation versus trapped air')
    ax.legend();ax.grid(alpha=.25);plt.show()

In [ ]:
# ILLUSTRATIVE model; parameters are not experimental estimates.
R_MODEL=1.0  # kPa s/µL
C_SYSTEM=0.01  # µL/kPa
P_GAUGE=20.0  # kPa
P_ATM=101.325  # kPa
F_PULSE=0.5  # Hz; illustrative
v_air=np.arange(0,101,2)
gas_final=v_air*P_ATM/(P_ATM+P_GAUGE)
C=C_SYSTEM+gas_final/(P_ATM+P_GAUGE)
tau=R_MODEL*C
H=1/np.sqrt(1+(2*np.pi*F_PULSE*tau)**2)
stored=C_SYSTEM*P_GAUGE+v_air-gas_final
fig,axs=plt.subplots(1,3,figsize=(13,3.5))
axs[0].plot(v_air,100*(1-H/H[0]));axs[0].set_ylabel('Predicted attenuation (%)')
axs[1].plot(v_air,tau);axs[1].set_ylabel('RC time constant (s)')
axs[2].plot(v_air,stored);axs[2].axhline(SYRINGE_CAPACITY_UL,ls='--',label='Nominal 80 µL');axs[2].legend(fontsize=7);axs[2].set_ylabel('Stored liquid (µL)')
for ax in axs:ax.set_xlabel('Initial air (µL)');ax.grid(alpha=.25)
fig.suptitle('Illustrative model: attenuation–settling–storage tradeoff');plt.tight_layout();plt.show()

## 4. What is the dead volume and transport delay?

**Measure:** tubing ID and length for each segment, plus tracer travel time or measured displaced volume. Separate **pre-mixing** volume from **post-mixing** volume. The chip estimate is $40\,mm\times0.25\,mm\times0.08\,mm=0.8\,µL$.

**Compute:** $t_{transport}\approx V_{post}/Q_{total}$. This is a nominal delay; dispersion can broaden the transition. Plot the volume budget and delay versus flow. Tubing dimensions below are placeholders.

In [ ]:
segments=pd.DataFrame([
    dict(name='PBS inlet tube',section='upstream',length_mm=100.,ID_mm=.5,shape='round'),
    dict(name='Ethanol inlet tube',section='upstream',length_mm=100.,ID_mm=.5,shape='round'),
    dict(name='Chip',section='post-mixing',length_mm=40.,ID_mm=np.nan,shape='rectangle',width_mm=.25,height_mm=.08),
    dict(name='Outlet tube',section='post-mixing',length_mm=100.,ID_mm=.5,shape='round')])
def volume_uL(r):
    area=np.pi*(r.ID_mm/2)**2 if r['shape']=='round' else r.width_mm*r.height_mm
    return area*r.length_mm # mm³ = µL
segments['volume_uL']=segments.apply(volume_uL,axis=1)
display(segments.round(3))
V_post=segments.loc[segments.section=='post-mixing','volume_uL'].sum()
Q_PBS=Q_PBS_UL_MIN;Q_ethanol=Q_ETHANOL_UL_MIN;Q_total=(Q_PBS+Q_ethanol)/60 # µL/s
transport_delay=V_post/Q_total
print(f'Post-mixing volume: {V_post:.3f} µL; delay: {transport_delay:.3f} s')
print(f'20 µL production time: {20/Q_total:.3f} s')
fig,ax=plt.subplots(figsize=(8,3.5));ax.barh(segments.name,segments.volume_uL)
ax.set(xlabel='Volume (µL)',title='Internal volume by segment (illustrative)');plt.tight_layout();plt.show()

In [ ]:
tracer_measurements=pd.DataFrame(columns=['segment','Q_measured_uL_min','delay_s'])
if len(tracer_measurements):
    tracer_measurements['estimated_uL']=tracer_measurements.Q_measured_uL_min*tracer_measurements.delay_s/60
    display(tracer_measurements)
else:print('Enter tracer measurements when available.')

In [ ]:
flow_grid=np.linspace(300,3000,100)
fig,ax=plt.subplots(figsize=(8,3.5));ax.plot(flow_grid,60*V_post/flow_grid)
ax.set(xlabel='Measured total flow (µL/min)',ylabel='Nominal post-mixing delay (s)',title='Transport delay vs flow — geometric estimate')
ax.grid(alpha=.25);plt.show()
print('At 3 mL/min, 20 µL takes 0.4 s; 40–60 µL takes 0.8–1.2 s, before dispersion/collection losses.')

## 5. When is the pressure sufficiently stable to collect?

**Question:** When do **both** pressure sensors meet a predefined mean-pressure tolerance and variability criterion over a 2-second **moving window**?

For each sensor, require $|\bar P-P_{target}|\leq\delta_P$ and $\sigma_P/|\bar P|\leq\epsilon$. Choose targets from measured plateau pressures, not arbitrary demonstration values. The candidate collection interval is shifted by post-mixing transport delay; actual arrival needs tracer validation.

**Plots:** pressure traces with stability windows and candidate collection windows. **The beep is a proposed trigger, not proof of a stable FRR.** At ~0.5 s CSV spacing a 2 s window has only ~4–5 readings; rapid pulsations remain unresolved.

In [ ]:
WINDOW_S=2.0
MEAN_TOL=0.05
CV_LIMIT=0.05
MIN_SAMPLES=4
TARGETS_KPA={} # e.g. {'P12159': 20, 'P22200': 10} after calibration
DISPERSION_MARGIN_S=0.0 # set from tracer measurements

def sensor_name(col):
    match=re.search(r'P\d+',col)
    return match.group(0) if match else col

def stability(df,pressure_cols,targets,window_s=WINDOW_S):
    a=df[['time_s']+pressure_cols].dropna().copy()
    a.index=pd.to_timedelta(a.time_s,unit='s')
    flags=[]
    for col in pressure_cols:
        name=sensor_name(col)
        if name not in targets:raise ValueError('Specify calibrated target for '+name)
        roll=a[col].rolling(f'{window_s}s',min_periods=MIN_SAMPLES)
        mean=roll.mean();cv=roll.std(ddof=0)/mean.abs().clip(lower=.001)
        a[name+'_mean']=mean;a[name+'_CV']=cv
        flags.append(((mean-targets[name]).abs()<=abs(targets[name])*MEAN_TOL)&(cv<=CV_LIMIT))
    a['stable']=np.logical_and.reduce([f.to_numpy() for f in flags])
    return a.reset_index(drop=True)

def intervals(a):
    f=a.stable.to_numpy(dtype=bool)
    starts=np.where(f&~np.r_[False,f[:-1]])[0]
    ends=np.where(f&~np.r_[f[1:],False])[0]
    return [(a.time_s.iloc[i],a.time_s.iloc[j]) for i,j in zip(starts,ends)]

if logs and TARGETS_KPA:
    for name,(df,ps,vs) in logs.items():
        try:a=stability(df,ps,TARGETS_KPA)
        except ValueError as exc:print(name,exc);continue
        periods=intervals(a);fig,ax=plt.subplots(figsize=(10,4))
        for col in ps:ax.plot(a.time_s,a[col],label=sensor_name(col))
        records=[]
        for j,(start,end) in enumerate(periods):
            ax.axvspan(start,end,alpha=.15,label='Pressure stable' if j==0 else None)
            collection_start=start+transport_delay+DISPERSION_MARGIN_S
            collection_end=end+transport_delay-DISPERSION_MARGIN_S
            if collection_end>collection_start:
                ax.axvspan(collection_start,collection_end,alpha=.12,label='Estimated collection' if j==0 else None)
            records.append(dict(start_s=start,end_s=end,duration_s=end-start,
                collection_start_s=collection_start,collection_end_s=collection_end,
                estimated_collectable_uL=max(0,collection_end-collection_start)*Q_total))
        ax.set(xlabel='Time (s)',ylabel='Pressure (kPa)',title=name+' — detector and collection delay')
        ax.legend();ax.grid(alpha=.25);plt.show()
        display(pd.DataFrame(records).round(3))
else:print('Set TARGETS_KPA using measured pressure plateaus to enable stability analysis.')

## Interpretation and next experiments

1. Confirm which pressure sensor corresponds to PBS and ethanol.
2. Record and annotate each run's actual flow settings, trapped air volume, refill events and chip geometry.
3. Measure gravimetric flow at multiple setpoints and fit branch-specific $R_h$ only where pressure plateaus exist.
4. Compare matched air-volume runs: steady pressure variability **and** settling time after refill.
5. Measure outlet dead volume and tracer dispersion; calibrate pressure targets and acceptance tolerances.
6. Validate the stability trigger against independent flow measurements and liposome DLS diameter/PDI.

**Important distinction:** A pressure-stable sample and a sample representative of the complete production cycle answer different research questions.